In [2]:
import os
import pandas as pd
import numpy as np
from docx import Document
from datetime import datetime

In [3]:
today_str = datetime.now().date().isoformat()

In [10]:
names_df = pd.read_excel('For Anthony.xlsx', sheet_name = '27.03.2023', names = ["Level", "Name", "Start", "End"])

In [20]:
names_df[names_df.Start.str.replace('\n', ' ') == "Course Start Date"].Name.apply(lambda x: x.split('\n')[:2]).values[0]

['Mon, Tue - Luke', 'Thur, Fri - John']

In [4]:
names_df = pd.read_excel('For Anthony.xlsx', names = ["Level", "Name", "Start", "End"])
for c in names_df.columns:
    names_df[c] = names_df[c].replace('\n', ' ', regex=True)
    names_df[c] = names_df[c].str.strip()
t_starts = np.where(names_df.Start == "Course Start Date")[0]
names_df.head()

,Level,Name,Start,End
0,Beginner,Teachers Here,Course Start Date,Course Finish Date
1,9.00-12.15 break 10.30-10.45 Room 8,John Smith,28.11.2022,09.06.2023
2,NaN,John Smith,09.01.2023,30.06.2023
3,NaN,John Smith,23.01.2023,14.07.2023
4,NaN,John Smith,03.10.2022,14.04.2023


In [5]:
filename = "Register - GE Morning (4).docx"
doc = Document(filename)

In [9]:
attend_tab = doc.tables[0]
prev_text = ''
level_str = 'CEFR level:'
for i, row in enumerate(attend_tab.rows[1:]):
        for j, cell in enumerate(row.cells):
            if cell.text:
                print(cell.text)

Week beginning:
Week beginning:
27.03.2023
Teacher/ Substitute:
Teacher/ Substitute:
Teacher/ Substitute:
Teacher/ Substitute:
Teacher/ Substitute:
Luke
Luke
Luke
Luke
Luke
Luke
Luke
Luke
Luke
Luke
Luke
Luke
Luke
          
          
CEFR level:
CEFR level:
Beginner
Teacher/ Substitute:
Teacher/ Substitute:
Teacher/ Substitute:
Teacher/ Substitute:
Teacher/ Substitute:
John
John
John
John
John
John
John
John
John
John
John
John
John
          
          
Full Name
Full Name
Full Name
Attendance
Attendance
Attendance
Attendance
Attendance
Homework
Homework
Homework
Homework
Homework
Weekly Performance*
Weekly Performance*
Weekly Performance*
Weekly Performance*
Weekly Performance*
Weekly Performance*
Weekly Performance*
Weekly Performance*
Test
Notes on learners
ü = present û = absent ¢ = late r = left early
ü = present û = absent ¢ = late r = left early
ü = present û = absent ¢ = late r = left early
M
T
W
T
F
M
T
W
T
F
V
P
G
C
W
L
R
* Grade learners weekly performance from 1-5 using t

In [56]:
def fill_template(class_i):
    if class_i == len(t_starts)-1:
        tmp_tab = names_df.iloc[t_starts[class_i]:]
    else:
        tmp_tab = names_df.iloc[t_starts[class_i]:t_starts[class_i+1]]

    tmp_tab     = tmp_tab.dropna(subset = ['Name'])
    group_level = tmp_tab.Level.values[0]
    group_room  = ''.join(tmp_tab.Level.values[1].split(' ')[3:])
    student_tab = tmp_tab.iloc[1:]

    attend_tab = doc.tables[0]
    prev_text = ''
    level_str = 'CEFR level:'
    for i, row in enumerate(attend_tab.rows[2:]):
            for j, cell in enumerate(row.cells):
                if cell.text:
                    if prev_text == level_str and cell.text != level_str:
                        cell.text = group_level
                    prev_text = cell.text
                    #print(cell.text)
    
    for i, row in enumerate(attend_tab.rows[5:]):
        for j, cell in enumerate(row.cells):
            if j == 1 and i in range(len(student_tab)):
                cell.text = student_tab.Name.values[i]

    

    save_str = f'{today_str}_{group_level}_{group_room}_Register.docx'
    doc.save(save_str)

In [57]:
for i in range(len(t_starts)):
    fill_template(i)